# Appendix A4 — R4 포트폴리오 구성 + Q-mode 통계 (Draft)

> **목적**: 논문 §5 결론 및 Appendix A4 작성을 위한 정량 근거 생성.
>
> - **셀 α**: R4 포트폴리오 비중 (저변동·고변동) by P × Q × Regime
> - **셀 β**: Q-mode별 Q값 통계 (평균, 부호 비율) by Regime
>
> **데이터 출처**: `final_pt/results/*.pkl` (108 LSTM + 108 ANN 슬롯)
>
> **검증 대상 가설**
> 1. anchor (mcap-mcap-fpm-pap)는 R4에서 저변동 < 고변동으로 strategy 본질이 뒤집힘
> 2. q_fpm은 R4에서 평균값·음수 비율 모두 다른 레짐 대비 극단적
> 3. p_w를 eq/rp로 바꾸면 R4 저변동 비중이 회복되는가? (사용자 가설)
> 4. 양수 제약 Q (lam/raw/inv/vsp)는 모든 레짐에서 양수 유지하는가? (모델 보증)

**주의**: 이 노트북은 `99_main_analysis.ipynb` / `99_analyze_ann_full.ipynb`와 충돌하지 않는 별도 draft이며, 결과를 확인한 뒤 Appendix A4 본문에 인용합니다.

## §0. 환경 셋업

In [1]:
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd

# 노트북이 appendix/에서 실행되든 final_pt/에서 실행되든 동작
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR if (NB_DIR / 'results').exists() else NB_DIR.parent
RESULTS_DIR = PROJECT_DIR / 'results'
assert RESULTS_DIR.exists(), f'results/ not found in {PROJECT_DIR}'

# 4-레짐 정의 (논문 기준 — master_table.py의 3-레짐과 별도)
REGIMES = [
    ('R1_회복',   '2010-01-01', '2012-06-30'),  # 30m — Post-GFC + EU 재정위기
    ('R2_확장',   '2012-07-01', '2019-12-31'),  # 90m — 장기 Bull
    ('R3_위기',   '2020-01-01', '2023-06-30'),  # 42m — COVID + 22 베어 + 인플레
    ('R4_정상화', '2023-07-01', '2025-12-31'),  # 30m — AI 랠리, Mag 7 집중
]

print(f'PROJECT_DIR : {PROJECT_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'pkl 파일 수 : {len(list(RESULTS_DIR.glob("*.pkl")))}')
print(f'레짐 수     : {len(REGIMES)}')

PROJECT_DIR : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/low_vol_with_lstm_har_bl/final_pt
RESULTS_DIR : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/low_vol_with_lstm_har_bl/final_pt/results
pkl 파일 수 : 216
레짐 수     : 4


## §1. 슬롯 파싱 및 pkl 로딩

`mat_{prior}_{p_w}_{q}_{omega}[_ann].pkl` 패턴 파싱.

pkl 내부 구조 (bl_runner.py 기준):
- `config`: 슬롯 설정 dict
- `ret`, `gross_ret`, `spy_ret`: 월별 수익률
- `weights`: T × N 종목 비중
- **`comp`**: T 행에 `low_weight`, `high_weight`, `n_eff`, `turnover` 등 — 본 노트북의 핵심 입력
- **`meta`**: T 행에 `Q`(view return), `lam`(위험회피) 등 — 본 노트북의 핵심 입력
- `errors`: walk-forward 에러 리스트

In [2]:
# mat_{prior}_{p_w}_{q}_{omega} 또는 끝에 _ann
PATTERN = re.compile(
    r'^mat_(?P<prior>[a-z]+)_(?P<p_w>[a-z]+)_(?P<q>[a-z]+)_(?P<omega>[a-z]+)(?P<ann>_ann)?\.pkl$'
)

def parse_slot(filename: str):
    m = PATTERN.match(filename)
    if not m:
        return None
    return {
        'prior': m.group('prior'),
        'p_w':   m.group('p_w'),
        'q':     m.group('q'),
        'omega': m.group('omega'),
        'model': 'ANN' if m.group('ann') else 'LSTM',
    }

slot_records = []
parse_fails  = []
load_fails   = []
for pkl_path in sorted(RESULTS_DIR.glob('*.pkl')):
    info = parse_slot(pkl_path.name)
    if info is None:
        parse_fails.append(pkl_path.name)
        continue
    try:
        with open(pkl_path, 'rb') as f:
            info['data'] = pickle.load(f)
    except Exception as e:
        load_fails.append((pkl_path.name, str(e)))
        continue
    info['path'] = pkl_path
    slot_records.append(info)

print(f'로드 성공 : {len(slot_records)} 슬롯')
print(f'parse 실패 : {len(parse_fails)} → {parse_fails[:5]}{"..." if len(parse_fails) > 5 else ""}')
print(f'load 실패  : {len(load_fails)}')

df_slots = pd.DataFrame([{k: v for k, v in s.items() if k not in ('data', 'path')} for s in slot_records])
print(f'\n모델별 슬롯 수:\n{df_slots["model"].value_counts().to_string()}')
print(f'\nq-mode 분포:\n{df_slots["q"].value_counts().to_string()}')
print(f'\np_w 분포:\n{df_slots["p_w"].value_counts().to_string()}')

로드 성공 : 216 슬롯
parse 실패 : 0 → []
load 실패  : 0

모델별 슬롯 수:
model
LSTM    108
ANN     108

q-mode 분포:
q
fix    36
fpm    36
inv    36
lam    36
raw    36
vsp    36

p_w 분포:
p_w
eq      72
mcap    72
rp      72


In [3]:
# 샘플 pkl 구조 확인 (anchor LSTM)
anchor_lstm = next(
    s for s in slot_records
    if s['prior'] == 'mcap' and s['p_w'] == 'mcap' and s['q'] == 'fpm' and s['omega'] == 'pap' and s['model'] == 'LSTM'
)
data = anchor_lstm['data']
print(f'샘플 슬롯: {anchor_lstm["path"].name}\n')
print(f'Keys: {list(data.keys())}\n')
for k, v in data.items():
    if isinstance(v, pd.DataFrame):
        print(f'  {k:<10}: DataFrame shape={v.shape}, cols={list(v.columns)[:8]}')
    elif isinstance(v, pd.Series):
        print(f'  {k:<10}: Series len={len(v)}')
    elif isinstance(v, dict):
        print(f'  {k:<10}: dict keys={list(v.keys())}')
    elif isinstance(v, list):
        print(f'  {k:<10}: list len={len(v)}')
    else:
        print(f'  {k:<10}: {type(v).__name__}')

# comp / meta 컬럼 확인
print(f'\ncomp columns : {list(data["comp"].columns) if "comp" in data and data["comp"] is not None else "N/A"}')
print(f'meta columns : {list(data["meta"].columns) if "meta" in data and data["meta"] is not None else "N/A"}')
print(f'\ncomp head:\n{data["comp"].head(3) if "comp" in data else "N/A"}')
print(f'\nmeta head:\n{data["meta"].head(3) if "meta" in data else "N/A"}')

샘플 슬롯: mat_mcap_mcap_fpm_pap.pkl

Keys: ['config', 'ret', 'gross_ret', 'spy_ret', 'weights', 'comp', 'meta', 'errors']

  config    : dict keys=['p_mode', 'p_weight', 'q_mode', 'q_value', 'omega_mode', 'prior', 'tc', 'max_weight', 'lstm_pred_path', 'name']
  ret       : Series len=192
  gross_ret : Series len=192
  spy_ret   : Series len=192
  weights   : DataFrame shape=(192, 592), cols=['A', 'AA', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABNB', 'ABT']
  comp      : DataFrame shape=(192, 10), cols=['n_stocks', 'eff_n', 'top10_share', 'top1_weight', 'top1_ticker', 'avg_vol', 'low_weight', 'high_weight']
  meta      : DataFrame shape=(192, 2), cols=['Q', 'lam']
  errors    : list len=0

comp columns : ['n_stocks', 'eff_n', 'top10_share', 'top1_weight', 'top1_ticker', 'avg_vol', 'low_weight', 'high_weight', 'turnover', 'tc_cost']
meta columns : ['Q', 'lam']

comp head:
            n_stocks      eff_n  top10_share  top1_weight top1_ticker  \
date                                                     

## §2. 셀 α — R4 포트폴리오 구성 (P × Q × Regime)

각 슬롯의 `comp` DataFrame에 저장된 월별 `low_weight`·`high_weight`를 레짐별로 평균.

**검증 대상**:
- anchor (mcap-mcap-fpm-pap)의 R4 비중이 "저변동 < 고변동"으로 뒤집혀 있는가?
- p_w를 eq/rp로 바꾸면 R4 저변동 비중이 회복되는가?
- 양수 제약 Q (lam/raw/inv/vsp)는 R4에서도 저변동 본질을 유지하는가?

In [4]:
def extract_regime_comp(slot: dict):
    """슬롯의 comp DataFrame에서 레짐별 평균 low_weight·high_weight 추출."""
    data = slot['data']
    comp = data.get('comp')
    if comp is None or len(comp) == 0:
        return []
    if 'low_weight' not in comp.columns or 'high_weight' not in comp.columns:
        return []
    rows = []
    for regime_label, start, end in REGIMES:
        mask = (comp.index >= start) & (comp.index <= end)
        if not mask.any():
            continue
        sub = comp.loc[mask]
        rows.append({
            'model':           slot['model'],
            'prior':           slot['prior'],
            'p_w':             slot['p_w'],
            'q':               slot['q'],
            'omega':           slot['omega'],
            'regime':          regime_label,
            'n_months':        int(mask.sum()),
            'low_weight_avg':  float(sub['low_weight'].mean()),
            'high_weight_avg': float(sub['high_weight'].mean()),
        })
    return rows

all_comp_rows = []
for slot in slot_records:
    all_comp_rows.extend(extract_regime_comp(slot))

df_comp = pd.DataFrame(all_comp_rows)
df_comp['spread'] = df_comp['low_weight_avg'] - df_comp['high_weight_avg']
print(f'전체 행 수 : {len(df_comp)}  (= {len(slot_records)} 슬롯 × {len(REGIMES)} 레짐)')
df_comp.head()

전체 행 수 : 864  (= 216 슬롯 × 4 레짐)


,model,prior,p_w,q,omega,regime,n_months,low_weight_avg,high_weight_avg,spread
0,LSTM,eq,eq,fix,he,R1_회복,30,0.815157,0.013619,0.801538
1,LSTM,eq,eq,fix,he,R2_확장,90,0.653031,0.045405,0.607626
2,LSTM,eq,eq,fix,he,R3_위기,42,0.668282,0.040395,0.627887
3,LSTM,eq,eq,fix,he,R4_정상화,30,0.580196,0.068221,0.511975
4,ANN,eq,eq,fix,he,R1_회복,30,0.739114,0.035952,0.703163


### §2.1 anchor 슬롯 (mcap-mcap-fpm-pap) — 레짐별 비중 추이

**핵심 질문**: R3 → R4에서 strategy가 어떻게 뒤집히는가?

In [5]:
anchor_mask = (
    (df_comp['prior'] == 'mcap') &
    (df_comp['p_w']   == 'mcap') &
    (df_comp['q']     == 'fpm')  &
    (df_comp['omega'] == 'pap')
)
anchor_pivot = df_comp[anchor_mask].pivot_table(
    index='regime',
    columns='model',
    values=['low_weight_avg', 'high_weight_avg', 'spread'],
).round(3)
print('Pyo & Lee anchor (mcap-mcap-fpm-pap) — 레짐별 비중')
print('=' * 80)
print(anchor_pivot.to_string())

Pyo & Lee anchor (mcap-mcap-fpm-pap) — 레짐별 비중
       high_weight_avg        low_weight_avg        spread       
model              ANN   LSTM            ANN   LSTM    ANN   LSTM
regime                                                           
R1_회복            0.082  0.087          0.598  0.556  0.516  0.469
R2_확장            0.181  0.205          0.453  0.431  0.272  0.226
R3_위기            0.198  0.212          0.420  0.403  0.222  0.190
R4_정상화           0.366  0.395          0.263  0.250 -0.103 -0.145


### §2.2 R4 정상화 — P × Q 조합별 비중 (anchor: prior=mcap, omega=pap)

**핵심 질문**: 
- p_w를 eq/rp로 바꾸면 R4 저변동 비중이 회복되는가?
- 양수 제약 Q (lam/raw/inv/vsp)는 R4에서도 저변동 본질을 유지하는가?

In [6]:
r4_mask = (
    (df_comp['prior']  == 'mcap') &
    (df_comp['omega']  == 'pap')  &
    (df_comp['regime'] == 'R4_정상화')
)
r4_sub = df_comp[r4_mask].copy()

# q-mode 순서 (양수 제약 → fpm)
Q_ORDER = ['lam', 'raw', 'inv', 'vsp', 'fpm']
r4_sub['q'] = pd.Categorical(r4_sub['q'], categories=Q_ORDER, ordered=True)

for model in ['LSTM', 'ANN']:
    sub_m = r4_sub[r4_sub['model'] == model]
    print(f'\n{"=" * 80}\n{model} — R4 정상화 비중 (행=p_w, 열=q)\n{"=" * 80}')
    for metric_label, col in [('저변동 비중', 'low_weight_avg'), ('고변동 비중', 'high_weight_avg'), ('spread (저-고)', 'spread')]:
        print(f'\n[{metric_label}]')
        pv = sub_m.pivot_table(index='p_w', columns='q', values=col).round(3)
        print(pv.to_string())


LSTM — R4 정상화 비중 (행=p_w, 열=q)

[저변동 비중]
q       lam    raw    inv    vsp    fpm
p_w                                    
eq    0.502  0.502  0.458  0.465  0.388
mcap  0.459  0.459  0.425  0.429  0.250
rp    0.499  0.499  0.456  0.464  0.392

[고변동 비중]
q       lam    raw    inv    vsp    fpm
p_w                                    
eq    0.167  0.167  0.191  0.187  0.242
mcap  0.170  0.170  0.196  0.193  0.395
rp    0.166  0.166  0.191  0.186  0.239

[spread (저-고)]
q       lam    raw    inv    vsp    fpm
p_w                                    
eq    0.335  0.335  0.267  0.278  0.146
mcap  0.290  0.290  0.229  0.236 -0.145
rp    0.332  0.332  0.265  0.277  0.153

ANN — R4 정상화 비중 (행=p_w, 열=q)

[저변동 비중]
q       lam    raw    inv    vsp    fpm
p_w                                    
eq    0.480  0.480  0.442  0.445  0.399
mcap  0.434  0.434  0.412  0.413  0.263
rp    0.478  0.478  0.441  0.445  0.407

[고변동 비중]
q       lam    raw    inv    vsp    fpm
p_w                                    
eq 

/var/folders/s9/dc7mvn1n1nl45gw342h0m_kr0000gn/T/ipykernel_43749/1274167226.py:17: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pv = sub_m.pivot_table(index='p_w', columns='q', values=col).round(3)
/var/folders/s9/dc7mvn1n1nl45gw342h0m_kr0000gn/T/ipykernel_43749/1274167226.py:17: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pv = sub_m.pivot_table(index='p_w', columns='q', values=col).round(3)
/var/folders/s9/dc7mvn1n1nl45gw342h0m_kr0000gn/T/ipykernel_43749/1274167226.py:17: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the 

### §2.3 R4 — fpm vs 양수 제약 Q 평균 비교 (p_w별)

양수 제약 4개 (lam/raw/inv/vsp)의 평균을 fpm 단일과 직접 대조.

In [7]:
r4_sub['q_class'] = r4_sub['q'].apply(lambda x: 'fpm' if x == 'fpm' else 'positive_avg')
r4_summary = (
    r4_sub.groupby(['model', 'p_w', 'q_class'])
    .agg(
        low_weight=('low_weight_avg', 'mean'),
        high_weight=('high_weight_avg', 'mean'),
        spread=('spread', 'mean'),
        n_slots=('q', 'count'),
    )
    .round(3)
)
print('R4 정상화 — fpm vs 양수 제약 Q 평균 비교')
print('=' * 80)
print(r4_summary.to_string())

R4 정상화 — fpm vs 양수 제약 Q 평균 비교
                         low_weight  high_weight  spread  n_slots
model p_w  q_class                                               
ANN   eq   fpm                0.399        0.234   0.165        1
           positive_avg       0.462        0.190   0.272        4
      mcap fpm                0.263        0.366  -0.103        1
           positive_avg       0.423        0.203   0.221        4
      rp   fpm                0.407        0.227   0.179        1
           positive_avg       0.460        0.189   0.271        4
LSTM  eq   fpm                0.388        0.242   0.146        1
           positive_avg       0.482        0.178   0.304        4
      mcap fpm                0.250        0.395  -0.145        1
           positive_avg       0.443        0.182   0.261        4
      rp   fpm                0.392        0.239   0.153        1
           positive_avg       0.479        0.177   0.302        4


## §3. 셀 β — Q-mode 통계 (Regime × Q-mode)

각 슬롯의 `meta['Q']` 시계열에서 레짐별 평균 q, 부호 비율 추출.

**검증 대상**:
- q_fpm의 R4 평균값·음수 비율이 다른 레짐 대비 얼마나 극단적인가?
- 양수 제약 Q (lam/raw/inv/vsp)는 모든 레짐에서 실제로 양수만 출력하는가?

In [8]:
def extract_regime_q(slot: dict):
    """슬롯의 meta DataFrame에서 레짐별 Q 통계 추출."""
    data = slot['data']
    meta = data.get('meta')
    if meta is None or 'Q' not in meta.columns:
        return []
    rows = []
    for regime_label, start, end in REGIMES:
        mask = (meta.index >= start) & (meta.index <= end)
        if not mask.any():
            continue
        Q = meta.loc[mask, 'Q'].dropna()
        if len(Q) == 0:
            continue
        rows.append({
            'model':    slot['model'],
            'prior':    slot['prior'],
            'p_w':      slot['p_w'],
            'q':        slot['q'],
            'omega':    slot['omega'],
            'regime':   regime_label,
            'n_months': len(Q),
            'mean_q':   float(Q.mean()),
            'median_q': float(Q.median()),
            'pct_neg':  float((Q < 0).mean()),
            'pct_pos':  float((Q > 0).mean()),
            'pct_zero': float((Q == 0).mean()),
        })
    return rows

all_q_rows = []
for slot in slot_records:
    all_q_rows.extend(extract_regime_q(slot))
df_q = pd.DataFrame(all_q_rows)
print(f'전체 Q 통계 행 수: {len(df_q)}')
df_q.head()

전체 Q 통계 행 수: 864


,model,prior,p_w,q,omega,regime,n_months,mean_q,median_q,pct_neg,pct_pos,pct_zero
0,LSTM,eq,eq,fix,he,R1_회복,30,0.003,0.003,0.0,1.0,0.0
1,LSTM,eq,eq,fix,he,R2_확장,90,0.003,0.003,0.0,1.0,0.0
2,LSTM,eq,eq,fix,he,R3_위기,42,0.003,0.003,0.0,1.0,0.0
3,LSTM,eq,eq,fix,he,R4_정상화,30,0.003,0.003,0.0,1.0,0.0
4,ANN,eq,eq,fix,he,R1_회복,30,0.003,0.003,0.0,1.0,0.0


### §3.1 anchor (prior=mcap, p_w=mcap, omega=pap) — Q-mode × Regime

기준 슬롯에서 q-mode별 Q 값 통계.

In [9]:
anchor_q_mask = (df_q['prior'] == 'mcap') & (df_q['p_w'] == 'mcap') & (df_q['omega'] == 'pap')
q_anchor = df_q[anchor_q_mask].copy()
q_anchor['q'] = pd.Categorical(q_anchor['q'], categories=Q_ORDER, ordered=True)

for model in ['LSTM', 'ANN']:
    sub_m = q_anchor[q_anchor['model'] == model]
    print(f'\n{"=" * 80}\n{model} — anchor 조건 (mcap-mcap-?-pap)\n{"=" * 80}')
    for metric_label, col in [('평균 Q 값', 'mean_q'), ('Q < 0 비율', 'pct_neg'), ('Q > 0 비율', 'pct_pos')]:
        print(f'\n[{metric_label}]')
        pv = sub_m.pivot_table(index='q', columns='regime', values=col).round(4)
        print(pv.to_string())


LSTM — anchor 조건 (mcap-mcap-?-pap)

[평균 Q 값]
regime   R1_회복   R2_확장   R3_위기  R4_정상화
q                                     
lam     0.0007  0.0076  0.0064  0.0051
raw     0.0004  0.0107  0.0064  0.0051
inv     0.0089  0.0017  0.0016  0.0018
vsp     0.0030  0.0024  0.0035  0.0022
fpm     0.0001 -0.0010 -0.0009 -0.0100

[Q < 0 비율]
regime   R1_회복   R2_확장   R3_위기  R4_정상화
q                                     
lam     0.0000  0.0000  0.0000     0.0
raw     0.0000  0.0000  0.0000     0.0
inv     0.0000  0.0000  0.0000     0.0
vsp     0.0000  0.0000  0.0000     0.0
fpm     0.4333  0.6111  0.6905     0.8

[Q > 0 비율]
regime   R1_회복   R2_확장   R3_위기  R4_정상화
q                                     
lam     1.0000  1.0000  1.0000     1.0
raw     0.7000  1.0000  1.0000     1.0
inv     1.0000  1.0000  1.0000     1.0
vsp     1.0000  1.0000  1.0000     1.0
fpm     0.5667  0.3889  0.3095     0.2

ANN — anchor 조건 (mcap-mcap-?-pap)

[평균 Q 값]
regime   R1_회복   R2_확장   R3_위기  R4_정상화
q                          

/var/folders/s9/dc7mvn1n1nl45gw342h0m_kr0000gn/T/ipykernel_43749/1035182905.py:10: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pv = sub_m.pivot_table(index='q', columns='regime', values=col).round(4)
/var/folders/s9/dc7mvn1n1nl45gw342h0m_kr0000gn/T/ipykernel_43749/1035182905.py:10: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pv = sub_m.pivot_table(index='q', columns='regime', values=col).round(4)
/var/folders/s9/dc7mvn1n1nl45gw342h0m_kr0000gn/T/ipykernel_43749/1035182905.py:10: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retai

### §3.2 q_fpm 부호 뒤집힘 — 전 (prior × p_w × omega) 조합 평균

q_fpm은 prior·p_w·omega 어떤 조합에서도 R4에서 동일한 부호 뒤집힘 패턴을 보이는가?

In [10]:
fpm_mask = df_q['q'] == 'fpm'
fpm_sub = df_q[fpm_mask].copy()

for model in ['LSTM', 'ANN']:
    sub_m = fpm_sub[fpm_sub['model'] == model]
    print(f'\n{"=" * 80}\n{model} — q_fpm의 레짐별 통계 (전 prior × p_w × omega 평균)\n{"=" * 80}')
    summary = sub_m.groupby('regime').agg(
        mean_q=('mean_q', 'mean'),
        pct_neg=('pct_neg', 'mean'),
        pct_pos=('pct_pos', 'mean'),
        n_slots=('mean_q', 'count'),
    ).round(5)
    print(summary.to_string())


LSTM — q_fpm의 레짐별 통계 (전 prior × p_w × omega 평균)
         mean_q  pct_neg  pct_pos  n_slots
regime                                    
R1_회복  -0.00187  0.73333  0.26667       18
R2_확장   0.00024  0.41852  0.58148       18
R3_위기   0.00085  0.46032  0.53968       18
R4_정상화 -0.00407  0.70000  0.30000       18

ANN — q_fpm의 레짐별 통계 (전 prior × p_w × omega 평균)
         mean_q  pct_neg  pct_pos  n_slots
regime                                    
R1_회복   0.00270  0.53333  0.46667       18
R2_확장   0.00103  0.31852  0.68148       18
R3_위기   0.00096  0.26190  0.73810       18
R4_정상화 -0.00366  0.78889  0.21111       18


### §3.3 양수 제약 Q — 모든 레짐에서 양수 유지 여부 확인 (모델 보증)

lam/raw/inv/vsp가 정말 양수만 출력하는지 검증. 만약 음수가 섞여 있다면 양수 제약 구현에 버그가 있다는 신호.

In [11]:
positive_mask = df_q['q'].isin(['lam', 'raw', 'inv', 'vsp'])
pos_sub = df_q[positive_mask].copy()
pos_summary = pos_sub.groupby(['q', 'regime']).agg(
    mean_q=('mean_q', 'mean'),
    min_pct_pos=('pct_pos', 'min'),
    max_pct_neg=('pct_neg', 'max'),
    n_slots=('mean_q', 'count'),
).round(5)
print('양수 제약 Q — 레짐별 통계 (전 prior × p_w × omega 평균)')
print('=' * 80)
print(pos_summary.to_string())
print('\n→ min_pct_pos가 1.0 이면 모든 슬롯·월에서 양수 출력 (보증)')
print('→ max_pct_neg가 0.0 이면 음수 한 번도 안 나옴 (보증)')

양수 제약 Q — 레짐별 통계 (전 prior × p_w × omega 평균)
             mean_q  min_pct_pos  max_pct_neg  n_slots
q   regime                                            
inv R1_회복   0.00895          1.0          0.0       36
    R2_확장   0.00175          1.0          0.0       36
    R3_위기   0.00158          1.0          0.0       36
    R4_정상화  0.00183          1.0          0.0       36
lam R1_회복   0.00070          1.0          0.0       36
    R2_확장   0.00762          1.0          0.0       36
    R3_위기   0.00639          1.0          0.0       36
    R4_정상화  0.00511          1.0          0.0       36
raw R1_회복   0.00040          0.7          0.0       36
    R2_확장   0.01065          1.0          0.0       36
    R3_위기   0.00641          1.0          0.0       36
    R4_정상화  0.00511          1.0          0.0       36
vsp R1_회복   0.00281          1.0          0.0       36
    R2_확장   0.00218          1.0          0.0       36
    R3_위기   0.00299          1.0          0.0       36
    R4_정상화  0.00204  

## §4. 종합 — Appendix A4 본문 작성 가이드

위 셀들의 출력을 보고 다음 핵심 발견을 본문에 인용:

### Appendix A4 작성 체크리스트

1. **§2.1 anchor 레짐별 비중**: R1 → R4 흐름에서 strategy가 언제부터 뒤집히는가?
   - R4 anchor LSTM/ANN 저변동·고변동 비중 인용

2. **§2.2 R4 P × Q 비중표**: 
   - p_w=mcap × q=fpm vs p_w=eq/rp × q=fpm 비교 → 사용자 가설 검증
   - p_w × q=양수제약 비교 → 양수 Q가 저변동 본질 유지 여부

3. **§2.3 fpm vs 양수 제약 평균**: p_w별로 두 분류 평균 직접 대조 → 핵심 contrast

4. **§3.1 anchor Q 통계**: q_fpm의 R1 → R4 mean_q와 pct_neg 추이 인용
   - 결론에 "R4에서 평균 −0.01, 음수 80%" 같은 수치 직접 인용

5. **§3.2 q_fpm 전 슬롯 평균**: 부호 뒤집힘이 anchor만의 현상이 아니라 모든 슬롯 평균에서도 동일한지 확인 → 일반성

6. **§3.3 양수 제약 보증**: 양수 Q들이 모두 양수만 출력함을 표로 보임 (모델 보증)

---

### 본 노트북에서 다루지 않은 항목 (별도 분석 필요)

- **셀 γ**: P × Q × Regime 2D grid for Q-effect dilution (결론에서는 제외했으나 §4.3 본문에 필요시)
- **개별 종목 분석**: R4에서 어떤 mega-cap 종목이 고변동으로 분류되어 어떤 비중으로 담겼는가 (필요시 weights DataFrame에서 추출)
- **레짐 전이 시점**: 2023-07 경계 전후 비중 변화 시각화 (필요시)